# 06 — Stage 1: Predictions & Country-Year Shadow Measure

Generate intervention probability predictions for all directed dyad-years and
aggregate to produce the country-year shadow measure.

**Inputs**:
- `data/interim/dd_spat_{cy}_{ud}.parquet` — feature data (from nb04)
- `data/interim/sl_model_{cy}_{ud}.pkl`   — fitted models (from nb05)
- `data/interim/sl_oof_{cy}_{ud}.parquet` — OOF predictions for onset rows

**Outputs**:
- `data/interim/cy_shadow_{cy}_{ud}.parquet` — country-year shadow variables:
  - `E_gov`, `E_opp` (raw expected-count sums)
  - `E_gov_asinh`, `E_opp_asinh` (asinh-transformed)
  - `E_gov_trim`, `E_opp_trim` (trimmed at tau = 0.001, asinh-transformed)

**Reference R scripts**: `15-generatePredictions.R`, `16-makePirate.R`

**Key design choice**: for Regan-period onset rows, the OOF predictions from
notebook 05 are used (avoiding train-on-predict leakage).  For all other rows,
the full-data model predicts directly.

**Cutpoint**: tau = 0.001 (tuned in Stage 2; see constructing.typ §Aggregating).

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve() / "src"))
from shadow.data.spatial import update_spatial_lags_proba, _build_W_cache

ROOT    = Path("..").resolve()
INTERIM = ROOT / "data" / "interim"

TAU         = 0.001   # cutpoint for trimming low-probability interveners
MAX_FP_ITER = 5       # fixed-point iterations for Nash-consistent predictions
FP_TOL      = 1e-4    # convergence: mean |Δ predicted probability|


In [ ]:
# Reuse feature column logic from nb05
ID_COLS = [
    "ccode_A", "ccode_B", "year", "ddyear",
    "onset_A", "regan_period", "intervention",
]

def get_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = set(ID_COLS)
    return [
        c for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]


def predict_proba_from_model(model_result: dict, X: np.ndarray) -> np.ndarray:
    """Use the fitted super-learner ensemble to predict P(class) for X."""
    scaler  = model_result["scaler"]
    pca     = model_result["pca"]
    weights = model_result["weights"]
    clfs    = model_result["classifiers"]

    X_sc = scaler.transform(X)
    X_pc = pca.transform(X_sc)

    proba = np.zeros((len(X), 3))
    for name, clf in clfs.items():
        proba += weights[name] * clf.predict_proba(X_pc)
    return proba


## Main loop: Nash-equilibrium predictions for all (B, A, t)

For each imputation, we solve the universal fixed-point equation:

    σ*(B, A, t) = M(X(B, A, t, σ*); θ)  for all (B, A, t)

The trained model θ is held fixed.  We iterate:
  predict → update spatial lags (all rows) → re-predict → … → converge.

This gives Nash-consistent predictions even for non-onset years: the
shadow measure E_gov(A, t) for a pre-conflict year reflects the counterfactual
equilibrium that would obtain if a conflict started, with all other potential
interveners playing their model-equilibrium strategies.


In [ ]:
for cy in range(1, 6):
    for ud in range(1, 6):
        print(f"── CY {cy}/5, UD {ud}/5 ", end="", flush=True)

        dd    = pd.read_parquet(INTERIM / f"dd_spat_{cy}_{ud}.parquet")
        model = joblib.load(INTERIM / f"sl_model_{cy}_{ud}.pkl")
        oof   = pd.read_parquet(INTERIM / f"sl_oof_{cy}_{ud}.parquet")

        feat_cols = get_feature_cols(dd)

        # Build W cache for ALL years (not just onset years): the universal
        # fixed-point updates spatial lags for every (A, t), onset or not.
        all_mask = pd.Series(True, index=dd.index)
        W_cache  = _build_W_cache(dd, all_mask)

        # ── Universal Nash fixed-point ─────────────────────────────────────
        # Solve σ*(B,A,t) = M(X(B,A,t,σ*); θ) for ALL (B,A,t).
        # The trained model θ is fixed; only the spatial lags change.
        prev_proba = None

        for fp_iter in range(MAX_FP_ITER):
            X_all     = dd[feat_cols].fillna(0).to_numpy(dtype=float)
            proba_all = predict_proba_from_model(model, X_all)

            if prev_proba is not None:
                delta = float(np.abs(proba_all - prev_proba).mean())
                if delta < FP_TOL:
                    break
            prev_proba = proba_all.copy()

            # Update spatial lags for ALL rows (onset_only=False).
            dd = update_spatial_lags_proba(
                dd,
                proba_all[:, 1],   # p_gov
                proba_all[:, 2],   # p_opp
                W_cache=W_cache,
                onset_only=False,
            )

        print(f"(FP: {fp_iter + 1} iters) ", end="", flush=True)

        # Overwrite Regan-period onset rows with OOF predictions (no leakage).
        dd["p_none"] = proba_all[:, 0]
        dd["p_gov"]  = proba_all[:, 1]
        dd["p_opp"]  = proba_all[:, 2]

        oof_keys = set(oof["ddyear"].values)
        oof_mask = dd["ddyear"].isin(oof_keys)
        if oof_mask.any():
            oof_lookup = oof.set_index("ddyear")[["p_none", "p_gov", "p_opp"]]
            for col in ["p_none", "p_gov", "p_opp"]:
                dd.loc[oof_mask, col] = (
                    dd.loc[oof_mask, "ddyear"].map(oof_lookup[col]).values
                )

        # Aggregate to country-year.
        def agg(group):
            pg = group["p_gov"].values
            po = group["p_opp"].values
            return pd.Series({
                "E_gov":      pg.sum(),
                "E_opp":      po.sum(),
                "E_gov_trim": pg[pg >= TAU].sum(),
                "E_opp_trim": po[po >= TAU].sum(),
                "n_B":        len(group),
            })

        cy_shadow = (
            dd.groupby(["ccode_A", "year"]).apply(agg, include_groups=False)
            .reset_index()
            .rename(columns={"ccode_A": "ccode"})
        )

        for col in ["E_gov", "E_opp", "E_gov_trim", "E_opp_trim"]:
            cy_shadow[f"{col}_asinh"] = np.arcsinh(cy_shadow[col])

        cy_shadow["cy_imp"] = cy
        cy_shadow["ud_imp"] = ud
        cy_shadow.to_parquet(INTERIM / f"cy_shadow_{cy}_{ud}.parquet", index=False)

        print(f"-> {len(cy_shadow):,} cy rows, "
              f"E_gov_asinh mean={cy_shadow['E_gov_asinh'].mean():.3f}")

n_files = len(list(INTERIM.glob("cy_shadow_*.parquet")))
print(f"\nShadow files produced: {n_files}  (expected 25)")
assert n_files == 25, f"Expected 25 files, got {n_files}"
